# 00 — Environment & Data Sanity Check

**Purpose of this push:** this is the first kernel pushed to Kaggle for this project. It does no modeling and requests no GPU. Its only job is to confirm two things before we spend any real effort or GPU quota:

1. The push -> run -> pull-logs workflow (`kaggle kernels push` / `status` / `output`) actually works end to end.
2. The real, Kaggle-mounted competition data matches the structure we inferred locally from the small metadata CSVs we pulled via the API (`train.csv`, `train_series.csv`, `test.csv`, `test_series.csv`, `sample_submission.csv` — no bulk image data was downloaded locally, per project rules in CLAUDE.md).

No conclusions from this notebook should be treated as modeling decisions — it's plumbing, not analysis.

## 1. Environment

In [ ]:
import sys
import platform

print("python:", sys.version)
print("platform:", platform.platform())

for pkg in ["numpy", "pandas", "torch", "pydicom"]:
    try:
        mod = __import__(pkg)
        print(f"{pkg}: {getattr(mod, '__version__', 'unknown version')}")
    except ImportError as e:
        print(f"{pkg}: NOT AVAILABLE ({e})")

In [ ]:
# We deliberately did not request a GPU for this push (enable_gpu: false in
# kernel-metadata.json) — this run is pure I/O/metadata checking, and GPU
# quota is a real shared constraint (see CLAUDE.md). Confirm that expectation:
try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
except ImportError:
    print("torch not available in this environment")

## 2. Top-level input structure

What does Kaggle actually mount under `/kaggle/input/` for this competition?

In [ ]:
import os

INPUT_ROOT = "/kaggle/input"
for root, dirs, files in os.walk(INPUT_ROOT):
    depth = root[len(INPUT_ROOT):].count(os.sep)
    if depth > 1:
        dirs[:] = []  # don't recurse further, just confirm top-level layout
        continue
    print(root)
    for f in sorted(files)[:10]:
        print("  ", f)
    if len(files) > 10:
        print(f"   ... ({len(files)} files total in this dir)")

In [ ]:
COMP_DIR = None
for name in os.listdir(INPUT_ROOT):
    if "knee" in name.lower():
        COMP_DIR = os.path.join(INPUT_ROOT, name)
print("Competition data dir:", COMP_DIR)
print(sorted(os.listdir(COMP_DIR)))

## 3. Metadata CSVs

Load the same files we already inspected locally (small pulls, not bulk data) and confirm the on-Kaggle versions match: same shapes, same columns, same label set.

In [ ]:
import pandas as pd

train = pd.read_csv(os.path.join(COMP_DIR, "train.csv"))
train_series = pd.read_csv(os.path.join(COMP_DIR, "train_series.csv"))
test = pd.read_csv(os.path.join(COMP_DIR, "test.csv"))
test_series = pd.read_csv(os.path.join(COMP_DIR, "test_series.csv"))
sample_submission = pd.read_csv(os.path.join(COMP_DIR, "sample_submission.csv"))

for name, df in [
    ("train", train),
    ("train_series", train_series),
    ("test", test),
    ("test_series", test_series),
    ("sample_submission", sample_submission),
]:
    print(f"{name}: shape={df.shape}")
    print("  columns:", list(df.columns))

In [ ]:
label_cols = [c for c in train.columns if c not in ("StudyInstanceUID", "Report")]
print("Label columns:", label_cols)

pos_rate = train[label_cols].apply(lambda s: pd.to_numeric(s, errors="coerce")).eq(1).mean().sort_values(ascending=False)
print("\nPositive rate per label (train.csv):")
print((pos_rate * 100).round(2).astype(str) + "%")

## 4. File-existence spot check

`train_series.csv` tells us which `(StudyInstanceUID, SeriesInstanceUID)` pairs exist, but not file paths directly. Confirm the assumed layout `train_series/{StudyInstanceUID}/{SeriesInstanceUID}/*.dcm` actually holds, on a handful of random series, and count slices per series.

In [ ]:
sample_rows = train_series.sample(n=5, random_state=0)

for _, row in sample_rows.iterrows():
    series_dir = os.path.join(
        COMP_DIR, "train_series", row["StudyInstanceUID"], row["SeriesInstanceUID"]
    )
    exists = os.path.isdir(series_dir)
    n_dcm = len([f for f in os.listdir(series_dir) if f.endswith(".dcm")]) if exists else 0
    print(f"exists={exists!s:5}  n_dcm={n_dcm:4d}  plane={row['Anatomical_Plane']:8s}  {series_dir}")

## 5. Summary

Fill in after the first successful run, then copy the relevant facts into `DESIGN.md` / `SESSION.md` — this notebook is scaffolding, not the record of truth.

- Push/pull workflow: (works / doesn't — note any issues)
- Train/test shapes matched local metadata pull: (yes/no)
- File layout assumption confirmed: (yes/no)
- Anything surprising: (...)